# Chapter 4 — The Context Window Is a Budget

## Question

**If the model advertises a large context window, how much of it is actually available for the task?**

Falsifiable version: after reserving output, reasoning, runtime additions, and a safety margin, is the usable input allowance strictly smaller than the headline — and can a session fit today while failing a projected tomorrow? The model below is fictional and synthetic, so the arithmetic cannot date.

## Setup — a fictional model and three budgets

Hard capacity is the outer wall. The usable budget subtracts reservations. The economic budget prices the bundle in abstract cost units — a labelled synthetic schedule, not provider pricing.

In [ ]:
from dataclasses import dataclass

HARD_CAPACITY = 16000
RESERVED_OUTPUT = 2000
REASONING_ALLOWANCE = 1000
SAFETY_MARGIN = 1000
USABLE_INPUT = HARD_CAPACITY - RESERVED_OUTPUT - REASONING_ALLOWANCE - SAFETY_MARGIN
print(f'hard capacity:       {HARD_CAPACITY}')
print(f'reserved output:     {RESERVED_OUTPUT}')
print(f'reasoning allowance: {REASONING_ALLOWANCE}')
print(f'safety margin:       {SAFETY_MARGIN}')
print(f'usable input budget: {USABLE_INPUT}')
assert USABLE_INPUT == 12000
assert USABLE_INPUT < HARD_CAPACITY

# Synthetic price schedule in abstract cost units per 1k tokens. NOT provider pricing.
PRICE = {'input': 1.0, 'output': 4.0}
print('\nEconomic schedule (synthetic cost units per 1k tokens):', PRICE)

## Baseline — one turn split into branches

Standing context recurs. Task context serves this turn. Accumulated context grows. Headroom is deliberately left empty.

In [ ]:
@dataclass
class BranchBudget:
    standing: int
    task: int
    accumulated: int

    @property
    def committed(self):
        return self.standing + self.task + self.accumulated

turn1 = BranchBudget(standing=2100, task=2900, accumulated=0)
print(f'standing:    {turn1.standing}')
print(f'task:        {turn1.task}')
print(f'accumulated: {turn1.accumulated}')
print(f'committed:   {turn1.committed} ({turn1.committed / USABLE_INPUT:.0%} of usable, {turn1.committed / HARD_CAPACITY:.0%} of headline)')
assert turn1.committed == 5000

## Intervention A — standing versus marginal cost

A 500-token tool schema sounds cheap. Paid every turn for forty turns, it exceeds a 10,000-token file loaded once. Same units, different dynamics.

In [ ]:
schema_per_turn, turns = 500, 40
standing_cumulative = schema_per_turn * turns
one_off_file = 10000
print(f'standing: {schema_per_turn} tokens x {turns} turns = {standing_cumulative} cumulative tokens')
print(f'marginal: {one_off_file} tokens x 1 turn = {one_off_file} tokens')
print(f'standing cost in units: {standing_cumulative / 1000 * PRICE["input"]:.0f}; one-off cost in units: {one_off_file / 1000 * PRICE["input"]:.0f}')
assert standing_cumulative == 20000
assert standing_cumulative > one_off_file

## Intervention B — an eight-turn trajectory

Standing recurs unchanged. Accumulated history and tool results compound. Nothing here behaves; the bundle only accrues.

In [ ]:
TASKS = [2900, 3100, 2800, 3300, 3000, 3400, 3200, 2900]
ACCUMULATED = [0, 900, 1800, 2700, 3600, 4500, 5700, 6600]
STANDING = 2100
trajectory = [BranchBudget(STANDING, task, acc) for task, acc in zip(TASKS, ACCUMULATED)]
print(f"{'turn':>4s} {'standing':>8s} {'task':>6s} {'accum':>6s} {'total':>6s} {'usable':>6s} {'headline':>8s}")
for i, branch in enumerate(trajectory, 1):
    print(f'{i:4d} {branch.standing:8d} {branch.task:6d} {branch.accumulated:6d} {branch.committed:6d} {branch.committed / USABLE_INPUT:5.0%} {branch.committed / HARD_CAPACITY:7.0%}')
last = trajectory[-1]
assert last.committed == 11600
standing_paid = STANDING * len(trajectory)
print(f'\nStanding content paid {len(trajectory)}x: {standing_paid} tokens for {STANDING} tokens of content.')
print(f'Accumulated share at turn 8: {last.accumulated / last.committed:.0%} (zero at turn 1).')

## Observation — fits now, fails later

Turn 8 fits the usable budget. The projection — a planning calculation, not a prediction — shows the trajectory borrowing against tomorrow.

In [ ]:
growth_per_turn = (trajectory[-1].accumulated - trajectory[0].accumulated) / (len(trajectory) - 1)
projected = last.committed + 3 * int(growth_per_turn)
print(f'fits current request (turn 8 within usable): {last.committed <= USABLE_INPUT}')
print(f'observed accumulation rate: ~{growth_per_turn:.0f} tokens/turn')
print(f'projected total after 3 more turns: ~{projected} vs usable {USABLE_INPUT}')
print(f'fits with projected next 3 turns: {projected <= USABLE_INPUT}')
print(f'headline story at turn 8: {last.committed / HARD_CAPACITY:.0%} full; budget story: {last.committed / USABLE_INPUT:.0%} committed and growing.')
assert last.committed <= USABLE_INPUT
assert projected > USABLE_INPUT

## Reconciliation — branches sum to instrument rows

In [ ]:
ledger = {'standing instructions': 1200, 'tool schemas': 500, 'project rules': 400,
          'current request': 300, 'current files': 2600,
          'history': 4100, 'tool results': 2500}
assert ledger['standing instructions'] + ledger['tool schemas'] + ledger['project rules'] == STANDING
assert ledger['current request'] + ledger['current files'] == last.task
task_ledger = ledger['current request'] + ledger['current files']
acc_ledger = ledger['history'] + ledger['tool results']
print(f'standing ledger: {STANDING}; task ledger: {task_ledger}; accumulated ledger: {acc_ledger}')
print(f'ledger total: {STANDING + task_ledger + acc_ledger} (turn-8-shaped bundle: matches by construction)')
session_input_tokens = sum(b.committed for b in trajectory)
session_units = session_input_tokens / 1000 * PRICE['input']
print(f'session input tokens across 8 turns: {session_input_tokens} (~{session_units:.0f} synthetic units)')

## Sensitivity — a larger safety margin moves the ceiling, not the dynamics

Policy change, same trajectory: which turn first exceeds usable when the margin doubles?

In [ ]:
margin2 = 2000
usable2 = HARD_CAPACITY - RESERVED_OUTPUT - REASONING_ALLOWANCE - margin2
first_exceed = next((n + 1 for n, b in enumerate(trajectory) if b.committed > usable2), None)
print(f'usable with doubled margin: {usable2}; first turn exceeding it: turn {first_exceed}')
assert usable2 == 11000 and first_exceed == 8


## Try it

1. Raise `SAFETY_MARGIN` to 2000 and find which turn first exceeds usable.
2. Double the accumulation step and watch the fits-now/fails-later gap widen.
3. Price the session with a different synthetic schedule and confirm the ranking of turns never changes — only the labelled units do.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print('double-growth projection:', last.committed + 3 * int(2 * growth_per_turn))

## What this demonstrates

- The headline overstates the input allowance: 16,000 hard capacity becomes 12,000 usable after explicit reservations.
- Standing costs recur per configuration while marginal costs are paid on admission; a small schema over forty turns outweighs a large one-off file.
- Accumulated context starts at zero and becomes the majority; the usable ceiling, not the headline, is the number to watch.
- A turn can fit today while the projection fails tomorrow.

## What this does not demonstrate

- That a bundle fitting the budget is useful, or that a larger window improves behaviour.
- That the synthetic price schedule resembles any provider's pricing.
- That fewer tokens prove lower real-world cost.
- Anything about behavioural degradation under load — that is Chapter 5.

## Connection to the chapter

The window is an outer capacity constraint; engineering allocates a smaller usable budget among competing claims. With accounting settled and headroom honoured, one question remains:

> If extra information fits comfortably, is including it harmless?

That is Chapter 5.